In [26]:
import numpy as np
import pandas as pd

import smplotlib
import matplotlib.pyplot as plt

from pathlib import Path
from matplotlib.lines import Line2D

import astropy.units as u
from astropy.io import fits
from astropy.table import Table, vstack
from IPython.display import display

from gordian.config import PhotConfig
from specconvert import convert_flux, convert_wave
from gordian.loading import load_phot_data
from sedpy.observate import getSED, load_filters


In [27]:
def create_synethetic_f150w2(flux_f115w, flux_f150w, flux_f200w, err_f115w, err_f150w, err_f200w):
    """Create synthetic F150W2 flux from F115W, F150W, and F200W fluxes."""

    flux_f150w2 = flux_f115w + flux_f150w + flux_f200w
    err_f150w2 = np.sqrt(err_f115w**2 + err_f150w**2 + err_f200w**2)

    return flux_f150w2, err_f150w2

In [28]:
F150W2_SYNTHETIC_FILTERS = ["F115W", "F150W", "F200W"]

FILTERS_TO_LOAD = ["F115W", "F150W", "F200W", "F277W","F444W"]

In [ ]:
# Load quiescent galaxy photometry data
LAPIS_DIR = Path("/Users/Jonah/PhD/Research/lapis")
phot_dir = LAPIS_DIR / "data_processed" / "photometry"

galaxy_files = {
    "cos-z5-q1": "cos-z5-q1_dja_v7_photometry.fits",
    "gs-z5-q1": "gs-z5-q1_dja_v7_photometry.fits",
    "1397": "1397_jades_dr5_photometry.fits",
    "8290": "8290_jades_dr5_photometry.fits",
    "8777": "8777_jades_dr5_photometry.fits",
    "72127": "72127_jades_dr5_photometry.fits",
    "80660": "80660_jades_dr5_photometry.fits",
    "gs-9209": "gs-9209_jades_dr5_photometry.fits",
}

In [35]:
# Create table of all photometry data
phot_tb = Table(
    names=[
        "ID",
        "F115W", "F115W_e",
        "F150W", "F150W_e",
        "F200W", "F200W_e",
        "F277W", "F277W_e",
        "F444W", "F444W_e",
    ],
    dtype=[
        "U20",
        float, float,
        float, float,
        float, float,
        float, float,
        float, float,
    ],
)

# Add fluxes to table
for galaxy_id, galaxy_file in galaxy_files.items():

    phot_config = PhotConfig(
        data_dir=phot_dir,
        data_file=galaxy_file,
        wave_col="WAVE_PIVOT",
        data_ext="DATA",
        err_col="ERR",
    )
    _, wave_pivots, flux_njy, err_njy, mask, filters = load_phot_data(phot_config)

    load_mask = [f in FILTERS_TO_LOAD for f in filters]
    flux_row = flux_njy[load_mask]
    err_row = err_njy[load_mask]
    phot_tb.add_row([str(galaxy_id)] + [x for pair in zip(flux_row, err_row) for x in pair])

In [ ]:
# Create synthetic F150W2 fluxes
flux_f150w2, err_f150w2 = create_synethetic_f150w2(
    flux_f115w=phot_tb["F115W"], 
    flux_f150w=phot_tb["F150W"], 
    flux_f200w=phot_tb["F200W"], 
    err_f115w=phot_tb["F115W_e"], 
    err_f150w=phot_tb["F150W_e"], 
    err_f200w=phot_tb["F200W_e"],
    )
phot_tb["F150W2"] = flux_f150w2
phot_tb["F150W2_e"] = err_f150w2